In [2]:
import unicodedata

import pandas as pd
from tensorflow.keras.utils import Sequence
from tensorflow.keras.layers import Conv2D,Dense,Dropout,Input,LSTM,Embedding
import os
import datasets
from datasets import Dataset,DatasetDict
import tensorflow as tf
import re
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [4]:
with open ("D:/project_deeplearning/TEP.en-fa.en",encoding="utf-8") as f:
    en_s=f.read().splitlines()

with open ("D:/project_deeplearning/TEP.en-fa.fa",encoding="utf-8") as f:
    fa_s=f.read().splitlines()


assert len(en_s)==len(fa_s)

df=pd.DataFrame(
    {
        "en":en_s,
        "fa":fa_s
    }
)

df=df.sample(n=30000,random_state=42)
df=df.reset_index(drop=True)

en=df["en"]
fa=df["fa"]
data=pd.DataFrame({"train":Dataset.from_pandas(df)})
print(data["train"][0])

{'en': 'stop her . somebody stop her reading .', 'fa': 'متوقفش كنيد يک نفر نگذاره اون ادامه بده .'}


In [5]:
train=data["train"]
train[:10]

0    {'en': 'stop her . somebody stop her reading ....
1        {'en': 'tetrastichous .', 'fa': 'چهاربيتي .'}
2    {'en': 'thats her stage name . i just said tha...
3    {'en': 'i wanna go . what .', 'fa': 'ميخوام بر...
4    {'en': 'im going to see the dragon warrior .',...
5    {'en': 'just think that i've received them and...
6    {'en': 'and the food was no different from a p...
7    {'en': 'are a couple jerkoffs .', 'fa': '2تا ا...
8    {'en': 'i think i should probably just stay wi...
9                {'en': 'vail .', 'fa': 'بکارخوردن .'}
Name: train, dtype: object

In [6]:
def unicode_to_asci(s):
    return "".join(c for c in unicodedata.normalize("NFC",s) if unicodedata.category(c)!='MN' )

In [7]:
len(data)

30000

In [8]:
def preprossing(w):
    w=unicode_to_asci(w.lower().strip())
    w=re.sub(r"([.!?])",r"\1",w)
    w=re.sub(r'([""])+',"",w)
    w=w.rstrip().strip()
    w="<start>"+w+"<end>"
    return w

In [9]:
en_sen="Im very happy."
preprossing(en_sen)

'<start>im very happy.<end>'

In [10]:
fa_sen="درود بر تو."
preprossing(fa_sen)

'<start>درود بر تو.<end>'

In [11]:
df["en"]=df["en"].apply(preprossing)
df["fa"]=df["fa"].apply(preprossing)

In [12]:
vocab_size=2000
max_length=256
batch_size=64

token_en=Tokenizer(num_words=vocab_size,filters="")
token_fa=Tokenizer(num_words=vocab_size,filters="")



token_en.fit_on_texts(df["en"])
token_fa.fit_on_texts(df["fa"])

en_seq=token_en.texts_to_sequences(df["en"])
fa_seq=token_fa.texts_to_sequences(df["fa"])

en_seq=pad_sequences(en_seq , maxlen=max_length,padding="post")
fa_seq=pad_sequences(fa_seq,maxlen=max_length,padding="post")

decoder_inputs_array=fa_seq[:,:-1]
decoder_targets_array=fa_seq[:,1:]

dataset = tf.data.Dataset.from_tensor_slices(((en_seq, decoder_inputs_array), decoder_targets_array))
dataset = dataset.shuffle(buffer_size=len(en_seq)).batch(batch_size).prefetch(tf.data.AUTOTUNE)



In [13]:
(x,dec_in),y=next(iter(dataset))
print(x.shape,dec_in.shape,y.shape)

(64, 256) (64, 255) (64, 255)


In [14]:
latent_dim=256

encoder_inputs=Input(shape=(max_length,),name="encoder_inputs")
encoder_embedding=Embedding(input_dim=vocab_size,output_dim=latent_dim,mask_zero=True)(encoder_inputs)
encoder_output,state_h,state_c=LSTM(latent_dim,return_state=True)(encoder_embedding)



decoder_inputs=Input(shape=(None,),name="decoder_inputs")
decoder_embedding=Embedding(input_dim=vocab_size,output_dim=latent_dim,mask_zero=True,name="decoder_embedding")
decoder_embedd=decoder_embedding(decoder_inputs)
decoder_lstm=LSTM(latent_dim,return_sequences=True,return_state=True,name="decoder_lstm")
decoder_outputs,state_h_dec,state_c_dec=decoder_lstm(decoder_embedd,initial_state=[state_h,state_c])




In [15]:
decoder_dense=Dense(vocab_size,activation="softmax")
decoder_outputs=decoder_dense(decoder_outputs)

In [16]:
model=tf.keras.Model([encoder_inputs,decoder_inputs],decoder_outputs)
model.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])

In [17]:
history=model.fit([en_seq,decoder_inputs_array],decoder_targets_array,
                  batch_size=batch_size,epochs=100)

Epoch 1/100
469/469 ━━━━━━━━━━━━━━━━━━━━ 378s 802ms/step - accuracy: 0.2838 - loss: 4.7778
Epoch 2/100
469/469 ━━━━━━━━━━━━━━━━━━━━ 445s 950ms/step - accuracy: 0.3134 - loss: 4.3831
Epoch 3/100
469/469 ━━━━━━━━━━━━━━━━━━━━ 452s 963ms/step - accuracy: 0.3263 - loss: 4.1723
Epoch 4/100
469/469 ━━━━━━━━━━━━━━━━━━━━ 478s 1s/step - accuracy: 0.3368 - loss: 4.0135
Epoch 5/100
469/469 ━━━━━━━━━━━━━━━━━━━━ 476s 1s/step - accuracy: 0.3471 - loss: 3.8724
Epoch 6/100
469/469 ━━━━━━━━━━━━━━━━━━━━ 478s 1s/step - accuracy: 0.3574 - loss: 3.7387
Epoch 7/100
469/469 ━━━━━━━━━━━━━━━━━━━━ 483s 1s/step - accuracy: 0.3674 - loss: 3.6066
Epoch 8/100
469/469 ━━━━━━━━━━━━━━━━━━━━ 440s 939ms/step - accuracy: 0.3774 - loss: 3.4783
Epoch 9/100
469/469 ━━━━━━━━━━━━━━━━━━━━ 448s 956ms/step - accuracy: 0.3872 - loss: 3.3506
Epoch 10/100
469/469 ━━━━━━━━━━━━━━━━━━━━ 473s 1s/step - accuracy: 0.3991 - loss: 3.2277
Epoch 11/100
469/469 ━━━━━━━━━━━━━━━━━━━━ 475s 1s/step - accuracy: 0.4106 - loss: 3.1045
Epoch 12/100
46

In [18]:
model.save('Translator.keras')